# 03 ReLU Bends One Coordinate

| | |
|---|---|
| **Path** | Multilayer perceptron |
| **Prerequisite** | 02 |

The rectifier, or ReLU, replaces a negative coordinate by zero and leaves a positive coordinate alone.

$$
\mathrm{ReLU}(z) =
\begin{cases}
z & \text{if } z > 0 \\
0 & \text{if } z \le 0
\end{cases}
$$

Its slope is $1$ on the positive side and $0$ on the negative side. At zero itself the slope is not unique. This path chooses $0$ there, so a unit sitting exactly on its kink sends no gradient backward.

Apply ReLU to each coordinate of $z$, then use the same output weights $(1, -2)$.

On three of the four gates, $z_2$ is already at least zero, so ReLU changes nothing. On gate 00, $z_2 = -1$, and ReLU turns that coordinate into $0$.


## Example 1 — Only gate 00 moves

$$
\begin{aligned}
(0, 0) &\mapsto \mathrm{ReLU}(0, -1) = (0, 0), & s &= 0 \\
(1, 0) &\mapsto (1, 0), & s &= 1 \\
(0, 1) &\mapsto (1, 0), & s &= 1 \\
(1, 1) &\mapsto (2, 1), & s &= 2 - 2 = 0
\end{aligned}
$$

The four scores are now $0, 1, 1, 0$. That is the target table. One bent coordinate did the whole job.


In [1]:
# ReLU changes only the negative coordinate. The four scores become the XOR targets.
def relu(value):
    return value if value > 0 else 0

gates = [(0, 0), (1, 0), (0, 1), (1, 1)]
hidden = []
scores = []
for x1, x2 in gates:
    h1 = relu(x1 + x2)
    h2 = relu(x1 + x2 - 1)
    hidden.append((h1, h2))
    scores.append(h1 - 2 * h2)
print(hidden)
print(scores)
assert hidden == [(0, 0), (1, 0), (1, 0), (2, 1)]
assert scores == [0, 1, 1, 0]


[(0, 0), (1, 0), (1, 0), (2, 1)]
[0, 1, 1, 0]


## Example 2 — The straight score and the bent score differ by 2 at the origin

Straight: $s(0, 0) = 2$. Bent: $s(0, 0) = 0$. The difference is exactly the piece ReLU removed,

$$
-2 \cdot \mathrm{ReLU}(-1) - \big(-2 \cdot (-1)\big) = 0 - 2 = -2
$$

so the bent score is the straight score plus $-2$.


In [2]:
# Removing the negative coordinate subtracts 2 from the straight score at the origin.
straight = 2
bent = 0
print(straight, bent, bent - straight)
assert bent - straight == -2


2 0 -2


## Pitfall — The kink has slope 0, including at zero

Gate 10 has $z_2 = 0$. ReLU leaves it at $0$, and the slope we chose there is also $0$. Later, when a derivative travels backward through that gate, the second hidden unit contributes nothing. Gate 11 has $z_2 = 1 > 0$, so the slope there is $1$.


In [3]:
# Slope 0 at the kink and on the left. Slope 1 on the right.
def relu_slope(value):
    return 1 if value > 0 else 0

print(relu_slope(-1), relu_slope(0), relu_slope(1))
assert (relu_slope(-1), relu_slope(0), relu_slope(1)) == (0, 0, 1)


0 0 1


## Summary

ReLU keeps the positive part of a score and replaces the rest by zero. With these weights it changes only gate 00, from a straight score of $2$ to a bent score of $0$. The four gates then read $0, 1, 1, 0$.
